## DAY 1 - OPENAI API

In [41]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from google import genai
from IPython.display import Markdown, display, update_display

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
g_api_key = os.getenv('GOOGLE_API_KEY')

In [19]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
openai = OpenAI(base_url=GEMINI_BASE_URL,api_key=g_api_key)
MODEL = "gemini-3.1-flash-lite"

In [ ]:
system_prompt = """You are an helpful assitant who help users to explain things clearly with neat examples. 
Analyse the problem and breakdown and breielfy explain the topics to the user. 
Don't provide the answer in Markdown
"""
user_prompt = """
    What is Quantization in AI?
"""

# Step 2: Make the messages list

messages = [{
    "role": "system", "content": system_prompt,
    "role": "user", "content": user_prompt
}]

# Step 3: Call OpenAI
response = openai.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages
)

# Step 4: print the result
display(Markdown(response.choices[0].message.content))

In the context of Artificial Intelligence, **Quantization** is the process of reducing the precision of the numbers (weights and biases) used to represent a neural network.

Think of it like **image compression**: just as you can reduce a high-resolution photo to a smaller file size by sacrificing a bit of color detail, you can shrink an AI model by reducing the mathematical precision of its internal calculations.

---

### How it works
Deep learning models typically run using **FP32** (Floating Point 32-bit) precision. This means every "weight" in the model is stored as a 32-bit number. 

Quantization converts these 32-bit numbers into smaller formats, such as:
*   **FP16/BF16 (16-bit):** Halves the size.
*   **INT8 (8-bit):** Reduces the size by 4x.
*   **INT4 (4-bit):** Reduces the size by 8x.

By moving from 32-bit to 8-bit, you aren’t changing the *structure* of the brain (the architecture); you are simply changing the *resolution* at which it stores its knowledge.

---

### Why do we use it? (The Benefits)

1.  **Reduced Memory Footprint:** A model that takes up 40GB of VRAM can be shrunk to 10GB or less. This allows massive models (like Llama 3) to run on consumer hardware like laptops or smartphones instead of needing massive data-center GPUs.
2.  **Faster Inference:** Because the computer has to process smaller numbers (8 bits vs 32 bits), the math happens much faster. This leads to higher "tokens per second" (faster text generation).
3.  **Lower Power Consumption:** Smaller calculations require less electricity, which is critical for mobile devices and edge hardware (like drones or IoT sensors).

---

### The Trade-off: Precision Loss
The primary drawback of quantization is **accuracy degradation**. 

Because you are rounding the numbers, the model effectively "loses nuance." If you quantize a model too aggressively (e.g., trying to compress it into 1-bit), the model might start hallucinating, lose its logic, or perform poorly on complex tasks. 

However, modern techniques like **QLoRA** and **GGUF** have made it possible to compress models significantly (down to 4-bit) with almost negligible impacts on intelligence.

---

### Real-World Analogy
Imagine you are a librarian trying to store 1,000 books in a tiny room. 
*   **FP32:** Storing the books in their full, hardcover, original format. It’s perfect, but you run out of space immediately.
*   **Quantization:** Making digital scans of the books or using a tiny, abbreviated font. You lose the original "feel" of the paper (precision), but you can now fit all 1,000 books in the room, and you can search through them much faster.

### Summary
Quantization is the **"secret sauce" that allows large AI models to run on your local computer.** Without it, the world of "Local LLMs" would not exist, as models would be too large and slow to run on anything other than enterprise-grade server hardware.

## Day 4 - Tokens and Tokenizer

In [ ]:
# Import tiktoken package from openai to validate how the tokens are splitted
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4")

In [ ]:
# Encode the words into tokens using encoding.encode
tokens = encoding.encode("Convert this sentence into tokens which can be feeded into LLM as the input.")
tokens

[12281,
 420,
 11914,
 1139,
 11460,
 902,
 649,
 387,
 5510,
 291,
 1139,
 445,
 11237,
 439,
 279,
 1988,
 13]

In [ ]:
# Display the unique tokens and their token_text mappings

for token in tokens:
    token_text = encoding.decode([token])
    print(f"{token}: {token_text}")

12281: Convert
420:  this
11914:  sentence
1139:  into
11460:  tokens
902:  which
649:  can
387:  be
5510:  feed
291: ed
1139:  into
445:  L
11237: LM
439:  as
279:  the
1988:  input
13: .


In [ ]:
# Validate the tokens for different words
print(encoding.decode([279]))
print(encoding.decode([55667]))

 the
collision


### Brochure provider for a website

In [44]:
from scraper import fetch_website_links, fetch_website_contents
import json

In [ ]:
# System Prompt for getting the relevant

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [ ]:
# This has the user_prompt along with the scrapped links

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
# API call to get the relevant links from the list of linsk from the website
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [30]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [ ]:
# Uses both the fetch content and website linsk from the scrapper and append it in a variable

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
# Creating a structired user prompt for the creating brochure based on content and links

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
# providing the website related input which we got earlier and feeding inot different llm model and streaming the output

def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemini-3.8-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
# Display the streamed results.

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the central collaboration platform and global home for the machine learning community. By empowering developers, researchers, and organizations to create, discover, and collaborate on state-of-the-art open-source AI, Hugging Face is shaping the future of technology.

---

## The Platform at a Glance

Hugging Face provides an end-to-end ecosystem where the entire machine learning lifecycle converges:

* **2M+ Models:** Discover, host, and fine-tune state-of-the-art models across natural language processing, computer vision, audio, and multimodal tasks.
* **500k+ Datasets:** High-quality, community-driven datasets to train and evaluate next-generation intelligence.
* **1M+ Spaces & Applications:** Showcase and test interactive AI demos directly in the browser, powered by flexible cloud compute and ZeroGPU technology.
* **Hugging Face Buckets & Infrastructure:** Persistent cloud storage, model versioning, and seamless tooling designed natively for ML workflows.
* **Research & Knowledge Hub:** Access Daily Papers, technical blogs, and community forums that track and define AI breakthroughs in real time.

---

## Solutions for Teams & Enterprise

Whether you are an ambitious startup or a global enterprise deploying AI at scale, Hugging Face provides commercial-grade infrastructure and tooling:

* **Hugging Face Enterprise & PRO:** Advanced privacy, dedicated compute, team management, and priority infrastructure for production workflows.
* **Inference Endpoints & Inference Providers:** Deploy models to dedicated, fully managed infrastructure on your cloud provider of choice with minimal latency and maximum scalability.
* **Enterprise Support:** Tailored technical guidance from the core engineers and researchers maintaining the world's most widely used open-source AI tools.

---

## Culture & Community: Open Source at Heart

At Hugging Face, openness, collaboration, and community ownership lead the way:

* **Open-Source DNA:** Hugging Face champions decentralized, accessible AI research—fostering transparent ecosystems where ideas are openly peer-reviewed and tested.
* **High-Velocity Innovation:** The team continuously ships cutting-edge research, from multi-harness reinforcement learning and foundation models to agentic coding tools like bug-fixing agents and native code reviewers.
* **Community First:** A global network spanning millions of researchers, students, and Fortune 500 engineers collaborating openly on GitHub, Discord, and Hugging Face Spaces.

---

## Careers & Building the Future

Joining Hugging Face means working at the epicenter of the global AI revolution:

* **Work with Purpose:** Help ensure the future of artificial intelligence remains open, transparent, and accessible to everyone—not locked behind closed garden walls.
* **Pioneer Frontier AI:** Collaborate with top ML engineers, research scientists, and open-source maintainers on tools that define modern AI development (such as the Transformers ecosystem, agentic harnesses, and scalable compute platforms).
* **Autonomy & Impact:** Thrive in an environment where direct contributions, technical craftsmanship, and rapid execution directly empower millions of developers worldwide.

---

**Explore the Hub, run an application, or join the movement:**  
Visit **huggingface.co** and start building with the global AI community.